# Sensibilidad institucional y de programa (R2-10)

Notebook independiente para responder al Revisor 2 (comentario 10): reporta concentración institucional/de programa por clúster (HHI) y hace un análisis leave-institution-out / leave-program-out sobre las 5 instituciones y 5 programas más grandes.

**Cómo correrlo:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno` → CPU (no necesita GPU).
2. Ajusta la ruta de tu `df_maestra.csv` en la celda de carga de datos si no está en `/content/drive/MyDrive/Proyecto/`.
3. Corre todas las celdas en orden (`Entorno de ejecución` → `Ejecutar todas`).
4. Los resultados se guardan automáticamente en tu Drive, en `/content/drive/MyDrive/Proyecto/sensibilidad_institucional/`, por si la sesión se desconecta.

In [ ]:
!pip install -q umap-learn

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
import umap.umap_ as umap_cpu
import warnings
warnings.filterwarnings("ignore")
print("✅ Librerías listas")

## 1. Cargar los datos desde Google Drive

Ajusta la ruta si tu `df_maestra.csv` está en otra carpeta de tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Proyecto/df_maestra.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed|^Column1')]
OUT_DIR = '/content/drive/MyDrive/Proyecto/sensibilidad_institucional'
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Shape original: {df.shape}")
df.head(3)

## 2. Preprocesamiento (idéntico al notebook original)

In [ ]:
def preprocesar_saber_pro(df_raw, sample_n=200_000, random_state=42):
    df_limpio = df_raw.copy()

    mapa_bano = {'1': 1, '2': 2, '3 o 4': 3, '5 o 6': 5, 'MAS DE 6': 6, 'NINGUNA': 0}
    mapa_estrato = {'Sin estrato': 0, 'Estrato 1': 1, 'Estrato 2': 2,
                     'Estrato 3': 3, 'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6}
    mapa_valormatricula = {
        'Sin costo': 0, 'Menos de 500 mil': 1,
        'Entre 500 mil y menos de 1 millón': 2,
        'Entre 1 millón y menos de 2.5 millones': 3,
        'Entre 2.5 millones y menos de 4 millones': 4,
        'Entre 4 millones y menos de 5.5 millones': 5,
        'Entre 5.5 millones y menos de 7 millones': 6,
        'Más de 7 millones': 7}
    mapa_educ = {
        'Ninguno': 0, 'Primaria incompleta': 1, 'Primaria completa': 2,
        'Secundaria (Bachillerato) incompleta': 3,
        'Secundaria (Bachillerato) completa': 4,
        'Técnica o tecnológica incompleta': 5,
        'Técnica o tecnológica completa': 6,
        'Educación profesional incompleta': 7,
        'EDUCACIÓN PROFESIONAL COMPLETA': 8, 'POSTGRADO': 9}
    mapeo_horas = {'0': 0, 'Menos de 10 horas': 1, 'Entre 11 y 20 horas': 2,
                    'Entre 21 y 30 horas': 3, 'Más de 30 horas': 4}
    mapeo_semestre = {str(i).zfill(2): i for i in range(1, 12)}
    mapeo_semestre['12 o más'] = 12

    mapeables = {
        'FAMI_CUANTOSCOMPARTEBAÑO':      mapa_bano,
        'FAMI_ESTRATOVIVIENDA':          mapa_estrato,
        'ESTU_VALORMATRICULAUNIVERSIDAD': mapa_valormatricula,
        'FAMI_EDUCACIONPADRE':           mapa_educ,
        'FAMI_EDUCACIONMADRE':           mapa_educ,
        'ESTU_HORASSEMANATRABAJA':       mapeo_horas,
        'ESTU_SEMESTRECURSA':            mapeo_semestre,
    }
    for col, mapa in mapeables.items():
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].map(mapa)

    columnas_puntaje = [
        'MOD_RAZONA_CUANTITAT_PUNT', 'MOD_LECTURA_CRITICA_PUNT',
        'MOD_COMPETEN_CIUDADA_PUNT', 'MOD_INGLES_PUNT', 'MOD_COMUNI_ESCRITA_PUNT']
    columnas_ordinales = [
        'FAMI_ESTRATOVIVIENDA', 'ESTU_VALORMATRICULAUNIVERSIDAD',
        'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE', 'ESTU_HORASSEMANATRABAJA']
    columnas_nominales = [
        'ESTU_TITULOOBTENIDOBACHILLER',
        'ESTU_PAGOMATRICULABECA', 'ESTU_PAGOMATRICULACREDITO',
        'ESTU_PAGOMATRICULAPADRES', 'ESTU_PAGOMATRICULAPROPIO',
        'ESTU_COMOCAPACITOEXAMENSB11',
        'FAMI_TIENEINTERNET', 'FAMI_TIENECOMPUTADOR',
        'FAMI_TIENEAUTOMOVIL', 'FAMI_TIENELAVADORA']
    col_geo = 'ESTU_COD_DEPTO_PRESENTACION'

    for col in columnas_puntaje:
        if col in df_limpio.columns:
            df_limpio[col] = pd.to_numeric(df_limpio[col], errors='coerce')
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mean())
    for col in columnas_ordinales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())
    for col in columnas_nominales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mode(dropna=True)[0])

    cols_usar = columnas_ordinales + columnas_puntaje + columnas_nominales
    cols_extra = [col_geo, 'INST_COD_INSTITUCION', 'ESTU_PRGM_ACADEMICO',
                  'PERIODO', 'PUNT_GLOBAL', 'ESTU_CONSECUTIVO']
    cols_df = cols_usar + [c for c in cols_extra if c in df_limpio.columns]
    df_filtrado = df_limpio[[c for c in cols_df if c in df_limpio.columns]].copy()
    df_filtrado = df_filtrado.dropna(subset=[c for c in columnas_puntaje if c in df_filtrado.columns])
    print(f"Filas después de limpieza: {len(df_filtrado):,}")

    cols_punt = [c for c in columnas_puntaje if c in df_filtrado.columns]
    cols_ord = [c for c in columnas_ordinales if c in df_filtrado.columns]
    cols_nom = [c for c in columnas_nominales if c in df_filtrado.columns]

    scaler = StandardScaler()
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_punt = scaler.fit_transform(df_filtrado[cols_punt])
    X_ohe = encoder.fit_transform(df_filtrado[cols_nom])
    X = np.hstack([df_filtrado[cols_ord].values, X_punt, X_ohe])
    feature_names = (cols_ord + list(scaler.get_feature_names_out(cols_punt))
                      + list(encoder.get_feature_names_out(cols_nom)))

    if np.isnan(X).any():
        from sklearn.impute import SimpleImputer
        X = SimpleImputer(strategy='median').fit_transform(X)
        print("NaN residuales imputados con mediana")

    if sample_n is not None and sample_n < df_filtrado.shape[0]:
        rng = np.random.default_rng(seed=random_state)
        idx = rng.choice(df_filtrado.shape[0], size=sample_n, replace=False)
        df_filtrado = df_filtrado.iloc[idx].reset_index(drop=True)
        X = X[idx]

    print(f"Preprocesamiento completo — shape X: {X.shape}")
    return df_limpio, df_filtrado, X, feature_names, encoder, scaler


In [ ]:
df_limpio_full, df_filtrado_full, X_full, feature_names, encoder, scaler = \
    preprocesar_saber_pro(df, sample_n=None)
n_total = X_full.shape[0]
print(f"Shape X_full: {X_full.shape}")

## 3. Partición 'publicada' de referencia (K=8)

In [ ]:
# Partición base K=8 (misma metodología del manuscrito: UMAP fit sobre
# 80,000 filas, transform sobre el resto; K-Means K=8), para tener una
# referencia 'publicada' propia sin depender de otros notebooks.
SEED_BASE = 42
K = 8
N_FIT = 80_000
N_EVAL = 50_000

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

rng_fit = np.random.default_rng(seed=SEED_BASE)
idx_fit = rng_fit.choice(n_total, size=N_FIT, replace=False)
rng_eval = np.random.default_rng(seed=0)
idx_eval = rng_eval.choice(n_total, size=N_EVAL, replace=False)

log("Ajustando UMAP base (K=8, seed=42)...")
reducer_base = umap_cpu.UMAP(n_components=2, random_state=SEED_BASE, n_neighbors=10,
                              low_memory=True, n_jobs=-1)
reducer_base.fit(X_full[idx_fit])
emb_eval_base = reducer_base.transform(X_full[idx_eval])
km_base = MiniBatchKMeans(n_clusters=K, random_state=SEED_BASE, n_init="auto", batch_size=10_000)
labels_pub = km_base.fit_predict(emb_eval_base)
log(f"Partición base lista. Tamaños: {np.bincount(labels_pub)}")

## 4. (a) Concentración institucional/de programa por clúster (HHI)

In [ ]:
def normalizar_programa(s):
    if pd.isna(s):
        return s
    s = s.upper().strip()
    return s.translate(str.maketrans('ÁÉÍÓÚ', 'AEIOU'))

def hhi(shares):
    return float(np.sum(np.square(shares)))

df_filtrado_full['PROGRAMA_NORM'] = df_filtrado_full['ESTU_PRGM_ACADEMICO'].apply(normalizar_programa)
df_eval = pd.DataFrame({
    'cluster': labels_pub,
    'institucion': df_filtrado_full['INST_COD_INSTITUCION'].values[idx_eval],
    'programa': df_filtrado_full['PROGRAMA_NORM'].values[idx_eval],
})
hhi_ref_inst = hhi(df_eval['institucion'].value_counts(normalize=True).values)
hhi_ref_prog = hhi(df_eval['programa'].value_counts(normalize=True).values)
print(f"HHI de referencia: instituciones={hhi_ref_inst:.4f}  programas={hhi_ref_prog:.4f}")

composicion = {}
for c in sorted(df_eval['cluster'].unique()):
    sub = df_eval[df_eval['cluster'] == c]
    inst_shares = sub['institucion'].value_counts(normalize=True)
    composicion[int(c)] = {
        'n': int(len(sub)), 'n_instituciones_distintas': int(sub['institucion'].nunique()),
        'share_institucion_top1': float(inst_shares.iloc[0]),
        'hhi_instituciones': hhi(inst_shares.values),
    }
    print(f"Clúster {c}: n={len(sub):,}  instituciones distintas={sub['institucion'].nunique()}  "
          f"HHI_inst={composicion[int(c)]['hhi_instituciones']:.4f} (ref {hhi_ref_inst:.4f})  "
          f"top1_inst={inst_shares.iloc[0]*100:.1f}%")

## 5. (b) Leave-institution-out / leave-program-out
⏱️ Reajusta el pipeline 10 veces (5 instituciones + 5 programas) — puede tardar 20-30 minutos.

In [ ]:
from sklearn.metrics import adjusted_rand_score

top_inst = df_filtrado_full['INST_COD_INSTITUCION'].value_counts().head(5).index.tolist()
top_prog = df_filtrado_full['PROGRAMA_NORM'].value_counts().head(5).index.tolist()
print(f"Top 5 instituciones a excluir: {top_inst}")
print(f"Top 5 programas a excluir: {top_prog}")

def correr_leave_out(mask_excluir, seed):
    idx_incluidos = np.where(~mask_excluir)[0]
    X_incl = X_full[idx_incluidos]
    n_incl = X_incl.shape[0]
    rng = np.random.default_rng(seed)
    idx_fit_local = rng.choice(n_incl, size=min(80_000, n_incl), replace=False)
    reducer = umap_cpu.UMAP(n_components=2, random_state=seed, n_neighbors=10,
                             low_memory=True, n_jobs=-1)
    reducer.fit(X_incl[idx_fit_local])
    mask_eval_incl = ~mask_excluir[idx_eval]
    idx_eval_incl = idx_eval[mask_eval_incl]
    emb_eval_lo = reducer.transform(X_full[idx_eval_incl])
    km = MiniBatchKMeans(n_clusters=K, random_state=seed, n_init="auto", batch_size=10_000)
    labels_new = km.fit_predict(emb_eval_lo)
    ari = adjusted_rand_score(labels_pub[mask_eval_incl], labels_new)
    return ari, int(mask_eval_incl.sum()), int(mask_excluir.sum())

resultados_sens = {"instituciones": {}, "programas": {}}
for i, inst in enumerate(top_inst):
    mask_excl = (df_filtrado_full['INST_COD_INSTITUCION'] == inst).values
    ari, n_eval_incl, n_excl = correr_leave_out(mask_excl, seed=800 + i)
    resultados_sens["instituciones"][str(inst)] = {'ari_vs_publicado': ari, 'n_excluidos': n_excl}
    print(f"[excluir institución {inst}] n_excluidos={n_excl:,}  ARI vs. publicado={ari:.4f}")

for i, prog in enumerate(top_prog):
    mask_excl = (df_filtrado_full['PROGRAMA_NORM'] == prog).values
    ari, n_eval_incl, n_excl = correr_leave_out(mask_excl, seed=850 + i)
    resultados_sens["programas"][str(prog)] = {'ari_vs_publicado': ari, 'n_excluidos': n_excl}
    print(f"[excluir programa {prog}] n_excluidos={n_excl:,}  ARI vs. publicado={ari:.4f}")

json.dump({'hhi_referencia': {'instituciones': hhi_ref_inst, 'programas': hhi_ref_prog},
           'composicion_por_cluster': composicion, 'sensibilidad_leave_out': resultados_sens},
          open(os.path.join(OUT_DIR, 'resultados.json'), 'w'), indent=2)
aris_todos = [v['ari_vs_publicado'] for v in resultados_sens['instituciones'].values()] + \
             [v['ari_vs_publicado'] for v in resultados_sens['programas'].values()]
print(f"\nARI leave-out: media={np.mean(aris_todos):.3f}  min={np.min(aris_todos):.3f}")
print("Valor esperado (manuscrito): media≈0.449±0.047 para 9/10 exclusiones, "
      "cae a ≈0.347 al excluir la institución más grande.")

## 6. Figura (Supplementary Figure S8)
La banda gris y la l\u00ednea punteada son la referencia de estabilidad del pipeline completo (ARI = 0.506 \u00b1 0.085), tomada del notebook `estabilidad_pipeline_colab.ipynb` -- no se recalcula aqu\u00ed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

insts = list(resultados_sens["instituciones"].items())
progs = list(resultados_sens["programas"].items())

labels = [f"Inst. {k}\n(n={v['n_excluidos']:,})" for k, v in insts] + \
         [f"{k.title()}\n(n={v['n_excluidos']:,})" for k, v in progs]
values = [v['ari_vs_publicado'] for k, v in insts] + [v['ari_vs_publicado'] for k, v in progs]
colors = ['#B33F3F'] * len(insts) + ['#3B6FA0'] * len(progs)

ARI_REF = 0.506
ARI_REF_SD = 0.085

fig, ax = plt.subplots(figsize=(13, 5.5), dpi=150)
ax.axhspan(ARI_REF - ARI_REF_SD, ARI_REF + ARI_REF_SD, color='lightgray', alpha=0.5)
ax.axhline(ARI_REF, color='gray', linestyle='--', linewidth=1, label=f'Ruido normal del pipeline (ARI={ARI_REF})')
bars = ax.bar(labels, values, color=colors)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
ax.set_ylabel('ARI vs. partici\u00f3n publicada')
ax.set_title('Sensibilidad leave-institution-out / leave-program-out\n(rojo=instituciones, azul=programas)')
ax.legend(loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figura_sensibilidad_institucional.png'), dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("\u2705 Figura guardada en Drive (Supplementary Figure S8)")
